# ⚠️ 실습 전 필수: 본인 드라이브에 사본 만들기!

상단 메뉴 **파일 → 드라이브에 사본 저장**을 클릭하세요.

사본을 만들지 않으면 작성한 코드가 저장되지 않습니다. 꼭 먼저 사본을 만든 뒤 시작하세요!

## 2주차 · LLM API 와 Agent 기초 (Responses API / Chat Completions)

지난 주에는 PyTorch로 딥러닝의 기본기를 다졌습니다. 이번 주에는 이미 학습이 끝난 거대 언어 모델(LLM)을
**API로 불러다 쓰는 법**과, 모델이 스스로 도구(tool)를 호출하며 문제를 풀어가는 **에이전트(Agent)의 기초**를 배웁니다.

### LLM을 API로 사용한다는 것

GPT와 같은 대형 모델은 수백 GB의 메모리를 요구하므로 개인 PC나 Colab에서 직접 실행하기 어렵습니다.
대신 OpenAI·구글 등은 자사 서버에 모델을 배포해 두고, **인터넷으로 요청을 보내면 응답을 돌려주는 창구**를 제공합니다.
이 창구가 **API**(Application Programming Interface)입니다.

> 🍽️ **식당에 비유하면:** 우리는 주문서(요청 = 프롬프트)를 건네고 요리(응답 = 생성된 텍스트)를 받습니다.
> 주방(모델 가중치, GPU)의 구조는 알 필요가 없습니다.
> 대신 **주문한 만큼 비용을 지불합니다.** LLM API는 주고받은 텍스트의 양(토큰)에 비례해 과금됩니다.

1주차가 요리를 직접 배우는 과정이었다면, 2주차는 이미 갖춰진 주방을 주문서 한 장으로 활용하는 과정에 해당합니다.

### 이 실습에서 쓰는 API

강의의 주제는 OpenAI의 [**Responses API**](https://platform.openai.com/docs/api-reference/responses) 입니다.
다만 Responses API는 유료 OpenAI 키가 필요하므로, 이 실습에서는 **무료로 실행 가능한 환경**을 함께 사용합니다.

- 🧪 **직접 실행하는 실습** → [**OpenRouter**](https://openrouter.ai) 의 무료 모델 `google/gemma-4-31b-it:free` 를 **Chat Completions API**로 사용합니다. (무료, 카드 등록 불필요)
- 📖 **개념 비교** → 같은 기능을 OpenAI **Responses API**로는 어떻게 쓰는지 코드로 나란히 보여줍니다.

두 API는 겉모습(입출력 형태)만 조금 다를 뿐, **function calling·에이전트의 핵심 개념은 동일**합니다.
그래서 무료 환경에서 배운 내용은 나중에 유료 Responses API로 그대로 옮겨 갈 수 있습니다.

### 오늘의 목표
1. OpenRouter 가입 & 무료 API 키 발급
2. LLM에 첫 요청 보내고 응답 받기 (Chat Completions)
3. Responses API와 Chat Completions의 차이 이해하기
4. 대화 맥락 이어가기
5. 모델에 **함수(도구)** 를 제공하고 호출하게 만들기 (function calling)
6. 함수 실행 결과를 되돌려주는 **에이전트 루프** 직접 구현하기

### 전체 흐름 개요

```
[나] "서울 날씨 알려주고 25×4도 계산해줘"
   │
   ▼
[LLM]  "get_weather(city='서울') 를 실행해줘"   ← 모델은 실행을 '요청'만 한다
   │
   ▼
[내 코드] get_weather("서울") 실행 → "맑음, 27도"   ← 실제 실행은 우리가 한다
   │
   ▼
[LLM]  "calculate('25*4') 도 실행해줘"  → 100
   │
   ▼
[LLM]  "서울은 맑고 27도이며, 25×4는 100입니다."   ← 더 부를 도구가 없으면 최종 답변
```

이 왕복을 **함수 호출이 없어질 때까지 반복하는 루프**가 곧 에이전트입니다. 9절에서 직접 만듭니다.

---
## 1. OpenRouter 가입 & API 키 발급

[**OpenRouter**](https://openrouter.ai) 는 하나의 API로 여러 회사의 LLM(OpenAI·구글·메타 등)을 골라 쓸 수 있게 해주는 중계 서비스입니다.
일부 모델은 **무료**로 제공되어, 이 실습처럼 학습·테스트 용도로 쓰기 좋습니다.

### 가입 및 키 발급 절차

1. [**openrouter.ai**](https://openrouter.ai) 접속 → 우측 상단 **Sign In** 클릭
2. **Google** 또는 **GitHub** 계정으로 로그인 (별도 회원가입 없이 소셜 로그인 가능)
3. 로그인 후 [**openrouter.ai/keys**](https://openrouter.ai/keys) 로 이동 (또는 우측 상단 프로필 → **Keys**)
4. **Create Key** 버튼 클릭 → 키 이름(예: `week2-practice`) 입력 → **Create**
5. 생성된 키(`sk-or-v1-...`)를 **복사**해 둡니다. ⚠️ 이 화면을 벗어나면 다시 볼 수 없으니 지금 복사하세요.

> 💳 **무료 모델과 한도:** `:free` 가 붙은 모델은 크레딧(결제) 없이 사용할 수 있습니다.
> 대신 사용량 제한이 있습니다 — 이 실습 모델 기준 대략 **분당 20회 / 하루 200회** 정도이며, 정책은 바뀔 수 있으니
> 정확한 한도는 [openrouter.ai](https://openrouter.ai/google/gemma-4-31b-it:free)에서 확인하세요. 실습에는 충분합니다.

---
## 2. 환경 설정

OpenRouter는 **OpenAI 파이썬 라이브러리와 호환**되도록 만들어졌습니다.
즉 OpenAI용으로 짠 코드에서 **접속 주소(`base_url`)와 키만 바꾸면** 그대로 동작합니다.
덕분에 나중에 유료 OpenAI 키가 생기면 두 줄만 고쳐서 옮겨갈 수 있습니다.

준비 순서는 세 단계입니다.

1. `openai` 라이브러리 설치
2. API 키를 환경변수에 넣기 (코드에 직접 적지 않기)
3. `client` 객체 만들기 — 앞으로 모든 요청은 이 `client` 를 통해 나갑니다

> `!pip install` 의 맨 앞 `!` 는 파이썬이 아니라 터미널 명령어로 실행하라는 Colab/Jupyter 문법입니다.
> `-q` 는 설치 로그를 줄이고, `--upgrade` 는 이미 설치돼 있어도 최신 버전으로 올립니다.
> Colab에 기본 설치된 `openai` 는 버전이 낮아 `tools` 인자를 지원하지 않을 수 있으므로 매번 최신으로 올립니다.


In [ ]:
!pip install -q --upgrade openai

### API 키 입력

방금 발급받은 OpenRouter 키를 파이썬이 읽을 수 있는 곳에 넣어 줍니다.

> 🔐 **API 키를 코드에 직접 작성하지 않는 이유**
> API 키는 **비밀번호이자 결제 수단**에 해당합니다. 노트북을 공유하거나 저장소에 올리면
> 제3자가 해당 키로 API를 호출할 수 있습니다. (공개 저장소에 노출된 키는 단시간에 수집됩니다.)
> 따라서 키는 코드가 아니라 **환경변수**(`os.environ`)로 관리하는 것이 원칙입니다.

아래 셀은 키를 두 가지 방법으로 찾습니다.

1. **Colab 보안 비밀**(왼쪽 사이드바 🔑 아이콘)에 `OPENROUTER_API_KEY` 로 저장해 두었다면 그 값을 사용합니다. 실행할 때마다 입력할 필요가 없습니다.
2. 저장된 값이 없으면 `getpass` 로 입력받습니다. `input()` 과 달리 **입력한 문자가 화면에 표시되지 않아** 화면 공유나 캡처 시에도 키가 노출되지 않습니다.

In [ ]:
import os
from getpass import getpass    # 입력한 글자를 화면에 표시하지 않는 안전한 입력 함수

if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        # Colab 왼쪽 🔑(보안 비밀) 메뉴에 저장해 뒀다면 그 값을 사용
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        # 저장해 둔 게 없거나 Colab이 아니면 직접 입력
        os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API 키(sk-or-v1-...)를 입력하세요: ")

print("API 키 설정 완료 ✅")   # 키 자체는 절대 print 하지 않습니다

### 클라이언트 만들기

`client` 는 어느 서버에 어떤 키로 접속할지를 담아 두는 객체입니다. 한 번 만들어 두면 이후 모든 요청에서 재사용합니다.

`base_url` 만 OpenRouter 주소로 바꾸면 OpenAI 라이브러리가 그대로 OpenRouter로 요청을 보냅니다. 이 줄을 지우면 OpenAI 본사로 나갑니다.

모델 이름 끝의 `:free` 는 무료 모델이라는 표시입니다. 모델 이름을 상수 `MODEL` 로 빼 두면 모델을 바꿀 때 이 한 줄만 수정하면 됩니다.


In [ ]:
from openai import OpenAI

client = OpenAI(
    # base_url 만 바꾸면 같은 코드가 OpenRouter로 나갑니다. (지우면 OpenAI 본사)
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# ⚠️ RateLimitError: Error code: 429 가 뜬다면(사용량 한도 초과) "openai/gpt-oss-20b:free" 로 바꿔 재시도하세요.
MODEL = "google/gemma-4-31b-it:free"

---
## 3. 첫 번째 응답 받기 (Chat Completions)

가장 기본형은 `client.chat.completions.create(...)` 입니다.
**대화 전체를 `messages` 라는 리스트로 넘기고**, 결과는 `response.choices[0].message.content` 로 꺼냅니다.

### `messages` 의 구조

`messages` 의 각 항목은 `role`(발화 주체)과 `content`(내용) 두 개의 키를 가진 딕셔너리입니다.

| role | 발화 주체 | 사용 시점 |
|---|---|---|
| `"system"` | 개발자가 모델에게 주는 **설정·규칙** | 말투, 역할, 지켜야 할 규칙 (5절) |
| `"user"` | 사용자(나)의 질문 | 실제 질문·요청 |
| `"assistant"` | 모델이 이전에 한 답변 | 대화를 이어갈 때 (6절) |

### 요청마다 대화 전체를 전송하는 이유

Chat Completions API는 **상태를 유지하지 않습니다(stateless)**.
서버는 각 요청을 독립적으로 처리하므로, 이전 대화 내용을 매번 `messages` 에 포함해 다시 전송해야
모델이 맥락을 파악할 수 있습니다. (6절에서 직접 다룹니다.)

### 응답 경로에 `choices[0]` 이 포함된 이유

`n=3` 과 같이 요청하면 후보 응답을 **여러 개** 받을 수 있기 때문에 목록 형태로 반환됩니다.
일반적으로 후보는 1개이므로 항상 `[0]` 번째를 사용합니다.

<!-- manim-visual -->
### 🎬 시각 자료: API 요청과 응답 한 번 왕복

내 노트북이 `messages` 를 담아 요청을 보내면 서버의 모델이 답변 하나를 돌려줍니다. 서버는 이 호출을 기억하지 않으며, 주고받은 토큰 수만큼 비용이 발생합니다.

![ApiRoundTrip](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/week2-responses-api-agent/week2/assets/ApiRoundTrip.gif)

> ManimGL로 제작 · gif는 GitHub raw URL에서 로드됩니다 (Colab 배포용)


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "자연어 처리에 대해 전혀 모르는 사람도 이해할 수 있게 두 문장으로 설명해줘."},
    ],
)

print(response.choices[0].message.content)

# 💡 content 를 바꿔 다른 질문으로도 실습해 보세요.
#    "딥러닝이 뭔지 초등학생도 이해할 수 있게 두 문장으로 설명해줘."
#    "다음 문장의 맞춤법을 고쳐줘: '오늘 회의는 3시에 시작되요.'"
#    "파이썬 리스트와 튜플의 차이를 표로 정리해줘." 

`response` 객체에는 답변 텍스트 외에도 여러 정보가 포함됩니다. 그중 두 가지를 확인합니다.

- **`response.model`** — 실제로 답한 모델 이름. OpenRouter는 요청한 모델이 붐비면
  같은 계열의 다른 버전으로 넘겨주기도 하므로, 실제 어떤 모델이 답했는지 확인하는 용도로 유용합니다.
- **`response.usage`** — **토큰 사용량**. LLM은 글자가 아니라 **토큰**(단어 조각) 단위로 텍스트를 처리하고,
  **요금도 토큰 수로 매겨집니다.**
  - `prompt_tokens` : 내가 보낸 입력의 토큰 수 (대화가 길어질수록 계속 커집니다)
  - `completion_tokens` : 모델이 생성한 답변의 토큰 수
  - `total_tokens` : 둘의 합 — 이 값이 곧 이번 호출의 비용입니다

> 📏 대략적인 기준으로 영어는 1토큰 ≈ 4글자, 한국어는 1글자 ≈ 1~2토큰입니다.
> 6절에서 대화를 이어갈 때 `prompt_tokens` 가 증가하는 양상을 함께 확인하면
> **대화 길이가 곧 비용**이라는 점을 수치로 확인할 수 있습니다.

In [ ]:
print("사용 모델:", response.model)          # 실제로 답한 모델 (요청한 것과 다를 수 있음)
print("토큰 사용량:", response.usage)        # prompt(입력) / completion(출력) / total
print("입력 토큰:", response.usage.prompt_tokens)

---
## 4. 📖 [참고] OpenAI Responses API 는 무엇이 다른가

강의의 주제인 **Responses API**는 OpenAI가 내놓은 최신 인터페이스로, 위의 Chat Completions보다 더 단순하게 설계되었습니다.
OpenAI 유료 키가 있다면 아래처럼 씁니다. **(OpenRouter에서는 지원하지 않아 이 실습에서는 실행하지 않습니다. 개념만 익혀두세요.)**

```python
from openai import OpenAI
client = OpenAI(api_key="OpenAI 키")          # base_url 없음 = OpenAI 본사

# 입력은 input=, 출력은 output_text 로 아주 간단
resp = client.responses.create(
    model="gpt-4.1",
    input="딥러닝을 두 문장으로 설명해줘.",
)
print(resp.output_text)

# 대화 이어가기도 이전 응답 id 한 줄이면 끝 (상태를 서버가 기억)
resp2 = client.responses.create(
    model="gpt-4.1",
    input="방금 설명을 한 문장으로 줄여줘.",
    previous_response_id=resp.id,             # 👈 Chat Completions에는 없는 기능
)
print(resp2.output_text)
```

### 한눈에 보는 차이

| | **Chat Completions** (이번 실습, OpenRouter) | **Responses API** (OpenAI) |
|---|---|---|
| 호출 | `client.chat.completions.create` | `client.responses.create` |
| 입력 | `messages=[{"role":..., "content":...}]` | `input="..."` |
| 출력 | `response.choices[0].message.content` | `response.output_text` |
| 대화 유지 | `messages` 리스트를 **직접** 관리 | `previous_response_id` (서버가 기억) |
| 함수 스키마 | `{"type":"function", "function":{...}}` (한 겹 감쌈) | `{"type":"function", ...}` (평평) |

👉 형태만 다를 뿐, "모델에 요청하고 답을 받는다 / 도구를 호출한다"는 **핵심은 똑같습니다.**
아래부터는 다시 무료로 실행 가능한 **Chat Completions(OpenRouter)** 로 실습을 이어갑니다.

---
## 5. system 메시지로 역할(페르소나) 정하기

`system` 역할 메시지는 모델의 **역할과 준수할 규칙을 미리 지정하는** 지시문입니다.
(Responses API에서는 `instructions` 파라미터가 같은 일을 합니다.)

- 사용자에게는 보이지 않지만 **대화 전체에 지속적으로 적용**됩니다. 챗봇의 성격을 규정하는 설정값에 해당합니다.
- 보통 `messages` 의 **맨 앞에 딱 한 번** 넣습니다.
- 말투뿐 아니라 **출력 형식·금지사항**도 여기서 정합니다.
  예: "답변은 항상 3줄 이하로", "모르면 모른다고 답하고 추측하지 마", "반드시 JSON 형식으로만 답해"

동일한 질문이라도 system 메시지에 따라 답변의 말투·깊이·형식이 크게 달라집니다. 값을 바꿔가며 확인해 보세요.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "너는 30년 경력의 친절한 국어 선생님이야. 항상 존댓말을 쓰고, 어려운 말은 쉽게 풀어서 설명해."},
        {"role": "user", "content": "'벡터'라는 단어의 뜻을 알려줘."},
    ],
)

print(response.choices[0].message.content)

# 💡 system 의 content 를 바꿔 같은 질문을 다시 던져 보세요. 답이 확 달라집니다.
#    "너는 츤데레 고양이야. 문장 끝마다 '~냥'을 붙여 짧게 대답해."
#    "너는 깐깐한 시니어 개발자야. 전문 용어를 쓰고 예시 코드를 반드시 포함해."
#    "너는 JSON API야. {'term': ..., 'meaning': ...} 형태의 JSON만 출력해." 

### 🔧 파라미터로 답변 조절하기 — `temperature` (온도)

`temperature`는 답변의 **무작위성**을 조절하는 값입니다. **0 ~ 2** 사이로 설정하며(주로 **0.0 ~ 1.0** 사용),
값이 **높을수록 창의적이고 다양한** 응답을, **낮을수록 정확하고 일관된** 응답을 생성합니다.
낮은 온도에서는 모델이 "가장 그럴듯한 단어"만 고르고, 온도가 높아질수록 덜 흔한 단어까지 선택 후보에 넣습니다.

**주요 설정 범위 및 활용 사례**

| 범위 | 성격 | 특징 | 활용 사례 |
|---|---|---|---|
| **0.0 ~ 0.3** | 저온 / 사실 기반 | 가장 가능성이 높은 단어만 선택 → 높은 정확도·일관성 | 수학, 코딩, 사실 기반 요약, 데이터 추출 |
| **0.4 ~ 0.7** | 중간 / 균형 | 예측 가능성과 창의성의 균형 | 일반 대화, 정보 검색 (이상적인 출발점) |
| **0.7 ~ 1.0** | 고온 / 창의적 | 덜 일반적인 단어의 선택 확률↑ | 브레인스토밍, 소설 작성, 마케팅 문구 |
| **1.0 이상** | 매우 높음 / 실험적 | 매우 무작위 → 엉뚱하거나 문맥에 안 맞는 결과 가능 | 실험적 용도 |

- `max_tokens` — 응답 길이(생성할 토큰 수)의 상한.

아래 셀의 `temperature` 를 **0.0 → 0.5 → 1.5** 로 바꿔가며 같은 프롬프트를 여러 번 실행하고,
답이 얼마나 달라지는지(0.0에서는 거의 매번 같은 답, 높을수록 매번 다른 답) 직접 비교해 보세요.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "'가을'을 주제로 한 줄짜리 시를 지어줘."}],
    temperature=1.2,      # 시라서 일부러 높게. 0에 가까울수록 매번 비슷한 답이 나옵니다.
    max_tokens=100,       # 답변 길이 상한. 너무 작으면 문장이 중간에서 끊깁니다.
)

print(response.choices[0].message.content)

# 💡 이 셀을 3번 실행한 뒤, temperature 를 0.0 으로 바꿔 다시 3번 실행해 비교해 보세요.
# 💡 content 를 "3+5는 얼마야? 숫자만 답해." 로 바꾸면 온도를 올려도 답이 거의 안 변합니다.

---
## 6. 대화 이어가기 — `messages` 직접 관리

3절에서 말했듯 Chat Completions는 **상태를 서버가 기억하지 않습니다.**
모델에는 기억이 없으며, **각 요청에서 전달받은 `messages` 가 모델이 참조할 수 있는 전부**입니다.

그래서 대화를 이어가려면 우리가 직접 기록을 관리해야 합니다.

```
1턴: [user1]                                    → 답변1
2턴: [user1, assistant1, user2]                 → 답변2
3턴: [user1, assistant1, user2, assistant2, user3] → 답변3
```

즉 **모델의 답변(`assistant`)까지 리스트에 다시 넣어 줘야** 다음 턴에서 맥락이 유지됩니다.

> 💸 **주의:** 대화가 길어질수록 매번 보내는 `messages` 가 커지고, `prompt_tokens` 도 함께 늘어납니다.
> 대화가 아주 길어지면 오래된 메시지를 잘라내거나 요약해서 넣는 방법을 씁니다.
>
> 📖 Responses API였다면 이 절 전체가 `previous_response_id=resp.id` **한 줄**로 끝납니다.
> 서버가 대화를 대신 기억해 주기 때문입니다.

<!-- manim-visual -->
### 🎬 시각 자료: 대화 맥락은 매번 다시 보낸다

턴이 늘어날 때마다 `messages` 리스트 전체를 다시 전송합니다. 모델의 답변(assistant)까지 쌓아야 맥락이 이어지고, 그만큼 prompt 토큰과 비용도 함께 늘어납니다.

![MessagesContext](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/week2-responses-api-agent/week2/assets/MessagesContext.gif)

> ManimGL로 제작 · gif는 GitHub raw URL에서 로드됩니다 (Colab 배포용)


In [ ]:
messages = [
    {"role": "user", "content": "내가 좋아하는 숫자는 7이야. 기억해둬."},
]

first = client.chat.completions.create(model=MODEL, messages=messages)
answer1 = first.choices[0].message.content
print("1차 응답:", answer1)

# ⭐ 모델의 답변도 기록에 넣어야 다음 턴에서 맥락이 유지됩니다. (누락하기 쉬운 부분)
messages.append({"role": "assistant", "content": answer1})

In [ ]:
# 새 질문을 덧붙이고 '기록 전체'를 다시 보냅니다. 이번 질문엔 '7'이 없지만 앞 대화로 알아냅니다.
messages.append({"role": "user", "content": "내가 좋아하는 숫자에 3을 곱하면 얼마야?"})

second = client.chat.completions.create(model=MODEL, messages=messages)
print("2차 응답:", second.choices[0].message.content)
print("이번 요청의 입력 토큰:", second.usage.prompt_tokens)   # 1턴보다 늘어난 만큼이 추가 비용

### 확인 실습

`messages` 에 이전 대화를 포함하지 않으면 모델은 "내가 좋아하는 숫자"를 알 수 없습니다.
아래와 같이 **기록 없이** 동일한 질문만 전송해 보고, 응답이 어떻게 달라지는지 비교해 보세요.

```python
# 기록을 빼고 두 번째 질문만 단독으로 보내기
alone = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "내가 좋아하는 숫자에 3을 곱하면 얼마야?"}],
)
print(alone.choices[0].message.content)   # "어떤 숫자인지 알려주세요" 같은 답이 나옵니다
```

이 실습이 보여주듯, **모델의 기억은 요청마다 우리가 구성해 주는 것**이며 모델 내부에 저장되지 않습니다.

---
## 7. 함수 호출(Function Calling) 기초

LLM은 학습한 시점까지의 지식만 가지고 있고, **바깥 세상과 연결되어 있지 않습니다.**
그래서 다음과 같은 작업은 스스로 수행하지 못합니다.

- 오늘 날씨, 지금 환율, 우리 회사 DB의 재고 수량 → **모르는 정보**
- 큰 수의 정확한 계산 → 그럴듯하게 **틀린 답**을 지어내기 쉬움 (환각, hallucination)
- 메일 보내기, 파일 저장하기 → **행동을 할 수단이 없음**

### 해결 방법: 도구 제공

파이썬 **함수(도구)** 를 정의하고 그 이름과 필요한 인자를 모델에 알려주면,
모델은 필요한 시점에 **해당 함수를 특정 인자로 실행해 달라고 요청**합니다.

> ⚠️ **핵심:** 모델은 함수를 **직접 실행하지 않습니다.**
> 모델의 역할은 어떤 함수를 어떤 인자로 호출할지 **결정해 전달하는 것**까지이며,
> **실제 실행은 전적으로 우리 코드가 담당**합니다. 따라서 위험한 함수는 제공하지 않으면 되고,
> 실행 전에 값을 검증하거나 차단할 수도 있습니다.

전체 흐름은 이렇습니다.

```
① 함수 만들기        →  ② 스키마로 모델에 소개  →  ③ 모델이 호출 요청(tool_calls)
                                                       ↓
⑤ 모델이 최종 답변   ←  ④ 우리가 실행 후 결과를 role:"tool" 로 전달
```

먼저 ① 단계로, 도구로 사용할 파이썬 함수를 정의합니다.
(실습 단계이므로 실제 날씨 API 대신 미리 정의한 데이터를 반환합니다.)

도구 함수는 어떤 값이 들어와도 예외를 내지 않고 문자열을 반환하도록 작성합니다.
모델이 등록되지 않은 도시를 넣을 수도 있는데, 그때 예외가 발생하면 뒤에서 만들 에이전트 루프 전체가 중단됩니다.


<!-- manim-visual -->
### 🎬 시각 자료: 함수 호출 5단계

질문 → 모델의 호출 요청(`tool_calls`) → 우리 코드가 실제 실행 → `role:"tool"` 로 결과 전달 → 최종 답변. 모델은 어떤 함수를 어떤 인자로 부를지 정할 뿐, 실행은 언제나 우리 코드가 합니다.

![FunctionCalling](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/week2-responses-api-agent/week2/assets/FunctionCalling.gif)

> ManimGL로 제작 · gif는 GitHub raw URL에서 로드됩니다 (Colab 배포용)


In [ ]:
def get_weather(city: str) -> str:
    # 실제라면 기상청 API를 부르겠지만, 실습이라 딕셔너리를 '가짜 DB'로 씁니다.
    fake_db = {
        "서울": "맑음, 27도",
        "부산": "흐림, 24도",
        "제주": "비, 22도",
    }
    return fake_db.get(city, f"{city}의 날씨 정보가 없습니다.")

# 모델에 연결하기 전에 함수 단독으로 테스트
print(get_weather("서울"))
print(get_weather("도쿄"))   # 없는 도시

# 💡 fake_db 에 데이터를 더 넣어 다른 데이터로도 실습해 보세요.
#        "대구": "맑음, 29도", "인천": "안개, 23도", "뉴욕": "맑음, 18도",
#
# 💡 아예 다른 주제의 도구를 만들어 보는 것도 좋습니다.
#    def get_exchange_rate(currency: str) -> str:
#        fake_rates = {"USD": "1,380원", "JPY": "890원(100엔)", "EUR": "1,490원"}
#        return fake_rates.get(currency, f"{currency} 환율 정보가 없습니다.")

### ② 도구 스키마 정의

모델은 파이썬 코드를 직접 볼 수 없습니다. 따라서 함수의 **이름, 기능, 필요한 인자**를
**JSON 스키마** 형식으로 기술해 전달해야 합니다. 도구의 사용설명서에 해당합니다.

각 항목의 의미는 다음과 같습니다.

| 키 | 의미 | 역할 |
|---|---|---|
| `name` | 함수 이름 | 모델이 이 이름으로 호출을 요청합니다. **실제 파이썬 함수명과 일치**시키는 것이 관리에 유리합니다 |
| `description` | 함수의 기능 | 모델이 **도구 사용 여부를 판단하는 유일한 근거**입니다 |
| `parameters` | 인자 명세 (JSON Schema) | 어떤 값을 어떤 타입으로 넣어야 하는지 |
| `properties` | 인자별 이름·타입·설명 | 설명에 **예시**를 포함하면 정확도가 향상됩니다 |
| `required` | 필수 인자 목록 | 명시하지 않으면 모델이 인자를 누락한 채 호출할 수 있습니다 |

> 💡 도구가 호출되지 않는 경우, 대부분 `description` 이 부실한 것이 원인입니다.
> "특정 도시의 현재 날씨를 알려준다" 와 같이 **기능과 사용 시점**을 구체적으로 기술하세요.

Chat Completions에서는 함수 정보를 `"function"` 키로 **한 번 감싸는** 형태를 씁니다.
(앞의 비교표에서 확인했듯 Responses API에는 이 계층이 없습니다.)

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",                          # 실제 파이썬 함수명과 똑같이
            "description": "특정 도시의 현재 날씨를 알려준다.",   # ⭐ 모델은 이 설명만 보고 도구 사용을 결정
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {                               # 함수의 매개변수 이름과 동일하게
                        "type": "string",
                        "description": "날씨를 조회할 도시 이름. 예: 서울, 부산",   # 예시를 넣으면 정확도가 오릅니다
                    },
                },
                "required": ["city"],
            },
        },
    }
]

# 💡 값을 정해진 것 중에서만 고르게 하려면 enum 을 씁니다.
#    "unit": {"type": "string", "enum": ["섭씨", "화씨"], "description": "온도 단위"}

### ③ 도구를 전달하고 질문하기

`tools=` 인자로 도구 설명서를 함께 전달해 질문합니다.
모델은 질문을 해석한 뒤 도구 사용 여부를 **스스로 판단**합니다.

- 도구 없이 답할 수 있는 질문(예: "안녕?") → 평소처럼 `content` 에 답변 텍스트가 옵니다.
- 도구가 필요한 질문(예: "서울 날씨 어때?") → `content` 는 보통 `None` 이 되고,
  대신 **함수 호출 요청**이 `message.tool_calls` 에 담겨 옵니다.

`tool_calls` 의 각 항목에는 세 가지가 들어 있습니다.

- `call.function.name` — 부를 함수 이름
- `call.function.arguments` — 인자. ⚠️ **딕셔너리가 아니라 JSON 문자열**입니다. (`'{"city": "서울"}'`)
- `call.id` — 호출의 고유 식별자. 결과를 반환할 때 **어느 요청에 대한 결과인지 연결하는 데** 사용합니다.

In [ ]:
messages = [{"role": "user", "content": "서울 날씨 어때?"}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,          # ⭐ 도구 설명서를 함께 전달 — function calling 의 시작
)

message = response.choices[0].message

print("일반 답변 content:", message.content)   # 도구를 부르려 할 땐 보통 None
print("함수 호출 요청 tool_calls:")
for call in (message.tool_calls or []):        # 도구를 안 부르면 None 이라 or [] 로 감쌉니다
    print("  호출할 함수:", call.function.name)
    print("  인자(문자열):", call.function.arguments)   # dict 가 아니라 JSON 문자열입니다
    print("  call id:", call.id)

# 💡 content 를 바꿔 모델이 언제 도구를 부르고 언제 안 부르는지 관찰해 보세요.
#    "안녕? 오늘 기분 어때?"            → 도구 없이 바로 답 (tool_calls 가 None)
#    "서울이랑 부산 날씨 둘 다 알려줘"   → tool_calls 가 2개 올 수 있습니다

---
## 8. ④⑤ 함수를 실행하고 결과를 모델에게 돌려주기

모델은 "`get_weather` 를 `city=서울` 로 실행해 달라"고 **요청**했을 뿐, 아직 최종 답변을 생성하지 않았습니다.
아직 날씨 값을 알지 못하기 때문입니다. 이어서 수행할 작업은 세 가지입니다.

1. 모델이 요청한 함수를 **실제로 실행**한다. (인자는 JSON 문자열이므로 `json.loads` 로 딕셔너리로 바꿔서)
2. 실행 결과를 **`role: "tool"` 메시지**로 만들어 대화에 이어 붙인다.
   이때 `tool_call_id` 에 앞서 받은 `call.id` 를 그대로 넣어 **요청과 결과를 연결합니다.**
3. 이 대화를 모델에게 **다시** 보내 자연어 최종 답변을 받는다.

즉 **한 번의 질문에 API 호출이 최소 두 번** 일어납니다. 이 왕복 구조가 function calling의 핵심입니다.

이 시점에 `messages` 는 아래처럼 쌓여 있게 됩니다.

```
[0] user      : "서울 날씨 어때?"
[1] assistant : (tool_calls=get_weather(city='서울'))   ← 모델의 호출 요청도 기록에 남긴다
[2] tool      : "맑음, 27도"  (tool_call_id=call_abc123)
```

> ⚠️ **주의:** 1번 단계에서 **모델의 호출 요청 메시지(`message`) 자체를 먼저 append** 해야 합니다.
> 이를 생략하고 `tool` 결과만 추가하면 요청 없이 결과만 존재하는 형태가 되어 API가 오류를 반환합니다.

In [ ]:
import json

# 1) 모델의 호출 요청 메시지를 먼저 기록에 추가
messages.append(message)

# 2) 요청된 함수를 실행하고 결과를 tool 메시지로 추가
for call in message.tool_calls:
    args = json.loads(call.function.arguments)      # JSON 문자열 -> 딕셔너리
    result = get_weather(**args)                    # get_weather(**{"city":"서울"}) == get_weather(city="서울")
    messages.append({
        "role": "tool",
        "tool_call_id": call.id,                    # 어떤 호출의 결과인지 연결 (필수)
        "content": result,                          # 결과는 반드시 문자열
    })

# 3) 도구 결과가 담긴 대화를 다시 전송 -> 최종 답변
final = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)
print(final.choices[0].message.content)

---
## 9. 에이전트(Agent) 루프 만들기

지금까지 한 과정을 정리하면 이렇습니다.

```
사용자 질문 → 모델 → (함수 호출 요청?) → 함수 실행 → 결과 전달 → 모델 → ...
```

8절에서는 이 왕복을 **수동으로 한 차례** 수행했습니다. 그러나 실제 질의는 한 번으로 끝나지 않습니다.
"서울과 부산 날씨를 알려주고 25×4도 계산해줘" 와 같은 질문이라면 모델은 도구를
**여러 차례, 여러 종류** 호출해야 하며, 앞선 도구의 결과에 따라 다음 도구를 결정하기도 합니다.

이 왕복을 **모델이 더 이상 함수를 호출하지 않을 때까지 반복**하는 루프로 감싸면,
모델이 스스로 도구를 선택해 문제를 해결하는 **에이전트**가 됩니다.
구조 자체는 20줄 남짓의 루프가 전부입니다.

### 에이전트를 만들 때 필요한 세 가지

| 준비물 | 하는 일 |
|---|---|
| `tools` | 모델에게 넘길 **도구 설명서 목록** (모델이 읽는 쪽) |
| `available_functions` | 이름 → 실제 파이썬 함수 **연결표**(dispatch table). 모델이 전달한 문자열 이름으로 함수를 찾기 위해 필요 |
| `max_turns` | **무한루프 방지 장치**. 모델이 도구 호출을 반복할 수 있으므로 반복 횟수에 상한을 둡니다 |

도구를 하나 더 추가합니다. 간단한 계산기입니다.
(LLM은 큰 수의 연산에서 오류를 내기 쉬우므로, 계산은 도구에 위임하는 것이 일반적입니다.)

> ⚠️ 아래 `calculate` 는 `eval` 을 사용합니다. `eval` 은 문자열을 파이썬 코드로 실행하므로
> 모델이 전달한 값을 그대로 넣으면 위험합니다. 허용 문자를 먼저 걸러내는 이유이며,
> 모델이 만들어 준 값도 사용자 입력과 동일하게 검증해야 합니다.


<!-- manim-visual -->
### 🎬 시각 자료: 에이전트 루프

도구 호출 요청이 없어질 때까지 같은 왕복을 반복합니다. 모델이 스스로 도구를 골라 쓰다가 더 부를 도구가 없으면 최종 답변을 내고 루프가 끝납니다.

![AgentLoop](https://raw.githubusercontent.com/SSU-NLP/Practical-AI-101/refs/heads/week2-responses-api-agent/week2/assets/AgentLoop.gif)

> ManimGL로 제작 · gif는 GitHub raw URL에서 로드됩니다 (Colab 배포용)


In [ ]:
def calculate(expression: str) -> str:
    # 실습용 최소 구현. 허용 문자만 통과시킨 뒤 계산합니다.
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "허용되지 않은 문자가 있습니다."
    try:
        return str(eval(expression))
    except Exception as e:
        # 도구는 예외를 밖으로 던지지 않습니다. 에러도 문자열로 주면 모델이 스스로 고쳐 재시도합니다.
        return f"계산 오류: {e}"


# 이름(문자열) -> 실제 함수 연결표. 모델은 "calculate" 라는 이름만 주기 때문에 필요합니다.
available_functions = {
    "get_weather": get_weather,     # () 를 붙이지 않습니다 — 실행이 아니라 함수 자체를 보관
    "calculate": calculate,
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "특정 도시의 현재 날씨를 알려준다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "도시 이름. 예: 서울"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "사칙연산 수식을 계산한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "계산할 수식. 예: '12 * 8'"},
                },
                "required": ["expression"],
            },
        },
    },
]

# 💡 도구를 더 만들어 보세요. 함수 정의 → available_functions 등록 → tools 스키마 추가, 3단계면 끝입니다.
#    def get_population(city: str) -> str:
#        fake_pop = {"서울": "938만 명", "부산": "329만 명", "제주": "67만 명"}
#        return fake_pop.get(city, f"{city}의 인구 정보가 없습니다.")
#    도구가 늘면 "서울 인구는 부산의 몇 배야?" 처럼 여러 도구를 연달아 써야 하는 질문도 풀립니다.

In [ ]:
import json

def run_agent(user_message, max_turns=5):
    # max_turns: 모델이 도구를 계속 부르며 맴돌 때를 대비한 무한루프 방지 장치.
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        message = response.choices[0].message
        messages.append(message)

        # 도구 호출 요청이 없다 = 할 일을 다 했다 -> 최종 답변
        if not message.tool_calls:
            return message.content

        for call in message.tool_calls:
            func = available_functions[call.function.name]   # 이름 -> 실제 함수
            args = json.loads(call.function.arguments)
            result = func(**args)
            print(f"  🔧 {call.function.name}({args}) -> {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })
        # 다음 바퀴로 돌아가 결과를 모델에게 보여주고 다음 행동을 정하게 합니다.

    return "(최대 반복 횟수를 초과했습니다.)"

# 💡 모델이 없는 함수 이름을 지어내면 KeyError 가 납니다. 아래처럼 막을 수 있습니다.
#    func = available_functions.get(call.function.name)
#    result = func(**args) if func else f"{call.function.name} 이라는 도구는 없습니다."


에이전트를 호출합니다. 아래 질문은 **날씨 조회(2회)와 계산(1회)** 을 모두 수행해야 답할 수 있습니다.
모델이 어떤 도구를 어떤 순서로 사용하는지 `🔧` 로그로 확인해 보세요.
특정 도구를 사용하라고 지시하지 않았음에도 **모델이 스스로 판단**한다는 점이 핵심입니다.

> ⚠️ 무료 모델은 성능이 제한적이라, 도구를 한 번에 완벽히 쓰지 못할 때도 있습니다.
> 도구를 호출하지 않거나 잘못된 인자를 전달한다면
> ① 질문을 더 명확히 작성하거나, ② `description` 을 구체적으로 수정하거나, ③ 여러 번 실행해 보세요.
> 이러한 실패를 관찰하고 원인을 찾아 고치는 과정도 실습의 일부입니다.

In [ ]:
# 도구 2개(get_weather, calculate)를 모두 써야 풀리는 질문입니다.
answer = run_agent("서울과 부산 날씨를 알려주고, 25 곱하기 4가 얼마인지도 계산해줘.")
print("\n최종 답변:\n", answer)

# 💡 다른 질문으로도 실습해 보세요. 🔧 로그로 도구 호출 횟수를 비교할 수 있습니다.
#    run_agent("안녕! 너는 누구야?")                → 도구를 아예 안 부름
#    run_agent("서울 기온에 2를 곱하면 몇 도야?")   → 날씨 결과를 계산 도구로 연결(도구 연쇄)

---
## 10. 연습문제 🎯

지금까지 에이전트가 쓰던 `get_weather` 는 미리 적어둔 **가짜 날씨**를 돌려주는 도구였습니다.
실제로 밖에 비가 와도 서울은 언제나 "맑음, 27도"였죠.

이번에는 이 도구를 **진짜 날씨 API**로 업그레이드해서,
에이전트가 지금 이 순간의 실제 데이터로 답하게 만들어 봅니다.

날씨 API는 [**Open-Meteo**](https://open-meteo.com) 를 사용합니다.
**가입도 API 키도 필요 없는** 무료 서비스라 바로 실습할 수 있습니다.

`# TODO` 로 표시된 부분만 하나씩 채우면 됩니다. 천천히 따라오세요!

### 문제 1. 진짜 날씨 도구 `get_real_weather` 를 에이전트에 추가하기

새 도구를 붙이는 순서는 7장에서와 언제나 같습니다.

**① 함수 만들기 → ② 이름표에 등록 → ③ 스키마 추가 → ④ 테스트**

<br>

**1단계. 함수 만들기**

아래 함수는 두 번의 API 호출로 실제 날씨를 가져옵니다.

1. 도시 이름으로 위도·경도를 찾고 (지오코딩)
2. 그 좌표의 현재 날씨를 조회합니다.

API 호출 부분은 미리 작성해 두었습니다.
여러분이 채울 곳은 마지막 `return` 한 줄입니다.

> ⭐ 도구가 반환한 문자열은 `role:"tool"` 메시지로 전달되고, 모델은 **그 문자열만 보고** 답을 만듭니다.
> `"23"` 만 주면 기온인지 습도인지 알 수 없으니,
> `"서울: 기온 23°C, 습도 60%"` 처럼 **이름표와 단위를 붙인 문장**으로 반환하세요.

In [ ]:
import requests

def get_real_weather(city: str) -> str:
    # Open-Meteo API로 도시의 '진짜' 현재 날씨를 조회한다. (무료, API 키 불필요)
    try:
        # 1) 도시 이름 -> 위도/경도 (한국어 이름도 가능)
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "ko"},
            timeout=5,
        ).json()
        if not geo.get("results"):
            return f"'{city}' 도시를 찾을 수 없습니다."
        loc = geo["results"][0]

        # 2) 위도/경도 -> 현재 날씨
        w = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": loc["latitude"],
                "longitude": loc["longitude"],
                "current": "temperature_2m,apparent_temperature,relative_humidity_2m,wind_speed_10m",
            },
            timeout=5,
        ).json()["current"]
    except requests.exceptions.RequestException as e:
        return f"네트워크 오류: {e}"

    # 사용 가능한 값:
    #   loc["name"]                -> 도시 이름 (예: "서울")
    #   w["temperature_2m"]        -> 기온 (°C)
    #   w["apparent_temperature"]  -> 체감 온도 (°C)
    #   w["relative_humidity_2m"]  -> 습도 (%)
    #   w["wind_speed_10m"]        -> 풍속 (km/h)

    # TODO: 위 값들을 이름표·단위가 붙은 한 문장으로 만들어 반환하세요.
    #   힌트: return f"{loc['name']}: 기온 {w['temperature_2m']}°C (체감 ...), 습도 ..., 풍속 ..."
    return  # <- 여기를 채우세요

# 진짜 오늘 날씨가 나오는지 확인해 보세요.
print(get_real_weather("서울"))
print(get_real_weather("Busan"))
print(get_real_weather("없는도시123"))   # 오류 처리 확인

# 💡 다른 도시로도 실습해 보세요. 지오코딩이 한국어·영어 이름을 모두 받습니다.
#    print(get_real_weather("제주"));  print(get_real_weather("New York"))

**2단계. 함수를 이름표에 등록**

모델은 함수를 직접 실행하지 못합니다.
"`get_real_weather` 를 실행해줘"라는 **이름(문자열)**을 보낼 뿐이죠.

그 이름으로 진짜 파이썬 함수를 찾아 실행하는 것이 `available_functions` 딕셔너리입니다.

새 도구를 등록하세요. 가짜 `get_weather` 는 헷갈리지 않게 치워 둡시다.

In [ ]:
# TODO: "get_real_weather" 라는 이름으로 get_real_weather 함수를 등록하세요.
#   힌트: available_functions["이름"] = 함수


# 가짜 날씨 도구 제거 (그대로 실행하면 됩니다)
available_functions.pop("get_weather", None)
tools[:] = [t for t in tools if t["function"]["name"] != "get_weather"]

# dict_keys(['calculate', 'get_real_weather']) 가 보여야 합니다.
print(available_functions.keys())

**3단계. 도구 스키마 추가**

모델은 우리가 짠 파이썬 코드를 보지 못합니다.
**스키마에 적은 설명만 읽고** 도구를 쓸지, 인자를 뭐라고 채울지 결정합니다.

채울 곳은 세 군데입니다.

| 채울 곳 | 역할 |
|---|---|
| `name` | 2단계 등록 이름과 **정확히** 동일해야 함 (다르면 `KeyError`) |
| `description` | 모델이 이 도구를 **언제** 쓸지 판단하는 기준 |
| `city` 의 `description` | 인자를 **어떻게** 채울지 안내 (예시 값을 함께 적기) |

In [ ]:
tools.append({
    "type": "function",
    "function": {
        "name": "",         # TODO ①: 2단계에서 등록한 이름과 똑같이
        "description": "",  # TODO ②: 예: "특정 도시의 실제 현재 날씨(기온·습도·풍속)를 조회한다."
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "",  # TODO ③: 예: "날씨를 조회할 도시 이름. 예: 서울, Busan"
                },
            },
            "required": ["city"],
        },
    },
})

# ['calculate', 'get_real_weather'] 가 보여야 합니다.
print([t["function"]["name"] for t in tools])

**4단계. 테스트!**

9장의 `run_agent` 는 `tools` 와 `available_functions` 를 그대로 쓰므로,
방금 추가한 새 도구를 바로 사용할 수 있습니다.

아래 질문은 날씨 조회와 계산을 **둘 다** 해야 답할 수 있습니다.
🔧 로그에 `get_real_weather` 가 찍히면 성공입니다. 창밖과 비교해 보세요! ☀️

In [ ]:
print(run_agent("서울의 지금 실제 날씨를 알려주고, 현재 기온에 2를 곱하면 얼마인지도 계산해줘."))

# 💡 다른 질문으로도 실습해 보세요.
#    run_agent("서울과 제주 중 어디가 더 습해?")        → 날씨 조회 2번 후 비교
#    run_agent("오늘 서울에서 우산이 필요할까?")        → 조회 값을 근거로 판단

### 문제 2. 에이전트 루프 복원하기

9장에서 만든 에이전트 루프의 흐름입니다.

```
질문 → 모델 → 도구 호출 요청? ── 없음 → ① 최종 답변, 종료
                │ 있음
                ▼
        ② 이름으로 진짜 함수를 찾아 실행
                ▼
        ③ 결과를 모델에게 반환 → 다시 모델에게
```

이번에는 이 루프를 직접 완성합니다.
그림의 ①②③ 세 곳이 빈칸(`________`)으로 비어 있습니다.

9장 코드를 그대로 베끼기보다, 각 빈칸이 **왜 필요한지** 생각하며 채워 보세요.

> ⚠️ 빈칸을 채우기 전에 실행하면 `NameError` 가 납니다. 정상이니 당황하지 마세요!

In [ ]:
import json

def run_agent_v2(user_message, system_prompt="", max_turns=5):
    messages = []
    if system_prompt:   # system 프롬프트가 있으면 맨 앞에 (문제 3에서 사용)
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    for turn in range(max_turns):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        messages.append(message)

        # TODO ①: 도구 호출 요청이 없으면 최종 답변. (힌트: message 의 어떤 속성이었나요?)
        if not ________:
            return message.content

        for call in message.tool_calls:
            # TODO ②: 모델이 보낸 이름으로 진짜 함수 찾기. (힌트: call.function.____)
            func = available_functions[________]
            args = json.loads(call.function.arguments)
            result = func(**args)
            print(f"  🔧 {call.function.name}({args}) -> {result}")

            # TODO ③: 결과를 모델에게 돌려주는 메시지입니다. role 과 tool_call_id 를 채우세요. (8장 참고)
            messages.append({
                "role": ________,
                "tool_call_id": ________,
                "content": str(result),
            })

    return "(최대 반복 횟수를 초과했습니다.)"

# 이 질문은 도구를 두 번 호출해야 답할 수 있습니다. 🔧 로그가 두 줄 찍히는지 관찰하세요.
print(run_agent_v2("서울이랑 부산 중에 지금 어디가 더 따뜻해?"))

### 문제 3. 기상 캐스터 페르소나 만들기

마지막 빈칸은 코드가 아니라 **프롬프트**입니다.

5장에서 배운 `system` 메시지는 말투뿐 아니라 에이전트의 **행동 규칙**을 정하는 곳이기도 합니다.
특히 조건 ③ "도구로 확인한 데이터만 말하기"는
에이전트가 정보를 지어내는 것을 막기 위해 실제 서비스에서도 널리 쓰이는 방법입니다.

In [ ]:
# TODO: 아래 세 조건을 모두 담은 system 프롬프트를 작성하세요.
#   ① 모든 답변을 '📺 날씨 캐스터입니다.' 로 시작하기
#   ② 날씨에 맞는 옷차림 조언을 한 문장 덧붙이기
#   ③ 도구로 조회한 실제 데이터만 근거로 말하고, 조회하지 않은 값은 지어내지 않기
weather_caster_prompt = ""   # <- 여기를 채우세요

# 답이 '📺 날씨 캐스터입니다.' 로 시작하고, 실제 날씨 + 옷차림 조언이 나오면 성공!
print(run_agent_v2("서울 날씨 어때? 오늘 뭐 입고 나갈까?", system_prompt=weather_caster_prompt))

# 💡 완성한 뒤 프롬프트를 바꿔 페르소나를 실험해 보세요.
#    "너는 등산 가이드다. 날씨를 확인한 뒤 오늘 산행이 적절한지 조언해라."
# 💡 조건 ③이 지켜지는지 보려면 조회하지 않은 값을 물어보세요.
#    run_agent_v2("서울의 내일 오후 강수 확률은?", system_prompt=weather_caster_prompt)

---
## 정리

### 이번 주에 배운 것

| 주제 | 핵심 |
|---|---|
| **OpenRouter** | `base_url` 만 바꾸면 OpenAI 라이브러리로 무료 모델을 쓸 수 있다 |
| **Chat Completions** | `messages` 로 요청하고 `choices[0].message.content` 로 응답을 받는다 |
| **Responses API** | `input` / `output_text` / `previous_response_id` — 더 단순하지만 개념은 동일 |
| **system 메시지** | 모델의 역할·말투·출력 형식을 정하는 보이지 않는 지시문 |
| **temperature** | 낮으면 일관적(사실·코드), 높으면 창의적(창작·아이디어) |
| **대화 맥락** | API는 상태를 유지하지 않는다. `assistant` 답변까지 직접 누적해야 맥락이 이어진다 |
| **Function calling** | 스키마로 도구 정의 → 모델의 `tool_calls` 요청 → **코드가 실행** → `role:"tool"` 로 결과 전달 |
| **에이전트 루프** | 도구 호출이 없어질 때까지 반복하는 구조 |
| **외부 API 연동** | Open-Meteo 실시간 날씨를 도구로 연결. 반환값의 형태, 스키마 설명, 행동 규칙이 에이전트 품질을 결정한다 |

### 반드시 기억할 세 가지

1. **모델은 함수를 실행하지 않는다.** 호출 대상을 결정할 뿐, 실행은 항상 코드가 담당한다.
2. **모델의 기억은 요청마다 구성된다.** `messages` 에 없는 정보는 모델에게 존재하지 않는다.
3. **도구의 `description` 이 에이전트의 성능을 좌우한다.** 설명이 부실하면 도구는 호출되지 않는다.

### 추가 학습 과제

- 연습문제의 Open-Meteo와 같이 **다른 공개 API**(환율·뉴스·대중교통 등)를 도구로 연결해 보기
- 도구를 3~4개로 늘리고, **여러 도구를 연달아 사용해야 풀리는 질문** 실험해 보기
- 동일한 코드를 **다른 모델**로 실행해 도구 사용 능력 비교하기
- 웹 검색·코드 실행 같은 **내장 도구(built-in tools)** 와 여러 에이전트를 엮는 **Agents SDK** 살펴보기

수고하셨습니다! 🎉